# 53-Qubit Google Supremacy Circuit: Tetron MBQC vs Direct Gate (Clifford / Stabilizer)

This notebook extends the 12-qubit validation study to the 53-qubit Google Sycamore
circuit, restricting to the **Clifford gate subset** so that the efficient
**stabilizer backend** can be used.

### What this notebook does

1. Load `circuit_n53_m12_s0_e0_pABCDCDAB.py` (Google's 53-qubit random circuit).
2. Keep only the Clifford gates:
   - Single-qubit: **√X** (`X**0.5`) and **√Y** (`Y**0.5`).
   - Two-qubit: **fSim(π/2, 0)** — the Clifford approximation where φ = 0.  
     All `FSimGate(θ, φ)` gates in the file are **replaced** by fSim(π/2, 0).
   - All other gates (**√W** = `PhasedXPowGate`, **Rz**, etc.) are **dropped**.
3. Translate every kept gate into its tetron MBQC equivalent using the
   validated decompositions from `Two_qubit_gate_supremacy.ipynb`:
   - fSim(π/2, 0) = **iSWAP†**, decomposed into H, S, CNOT primitives.
   - Classical bits are **re-used** via an **8-bit scratch register**
     (no ever-growing classical memory — each gate applies its Pauli
     corrections immediately and the bits are available for the next gate).
4. Build a **106-qubit MBQC circuit** (53 data + 53 ancilla tetrons) using
   `qubit_mapping_53.py`.
5. Build a **53-qubit direct reference circuit** using stabilizer-native Qiskit gates.
6. Run both circuits with `AerSimulator(method='stabilizer')` and compare
   output bitstring distributions via Hellinger fidelity.

### Why the comparison differs from the 12-qubit notebook

For 53 qubits the statevector has 2⁵³ ≈ 9 × 10¹⁵ entries — far beyond RAM.
The stabilizer tableau represents the same state in **O(n²)** space.
Comparison is therefore shot-based: if the MBQC feed-forward rules are correct,
the marginal distribution on the 53 logical data qubits should match the
direct-circuit distribution.

In [ ]:
import os, sys, importlib.util

import numpy as np
import matplotlib.pyplot as plt
import cirq

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import iSwapGate
from qiskit.circuit.classical import expr
from qiskit_aer import AerSimulator

REPO_ROOT  = os.path.abspath('.')
TETRON_DIR = os.path.join(REPO_ROOT, 'src', 'tetron')
GOOGLE_DIR = os.path.join(REPO_ROOT, 'google_53qubits_circuit')

if TETRON_DIR not in sys.path:
    sys.path.insert(0, TETRON_DIR)

from qubit_mapping_53 import (
    grid_to_qiskit_index,
    grid_to_sq_ancilla_index,
    grid_edge_to_qiskit_indices,
    GRID_TO_LOGICAL,
)

print('Imports OK.  Tetron dir:', TETRON_DIR)
print('Google dir: ', GOOGLE_DIR)

## 2. MBQC helper functions

These are the parity-measurement primitives and MBQC gate builders validated
in `Two_qubit_gate_supremacy.ipynb`.  They all share a single **8-bit scratch
register** — each function writes its measurement outcomes to specific bit
positions, applies the Pauli corrections immediately, then returns (so those
bits can be safely overwritten by the next call).

Bit-position layout of the scratch register:
- `scratch[0:5]` — single-qubit gates (H, S, SH, HS, HSH)
- `scratch[5:8]` — MBQC CNOT (non-overlapping with single-qubit slots)

In [ ]:
# ---------------------------------------------------------------------------
# Parity-measurement primitives
# Convention: Y = S X S†  (consistent with Two_qubit_gate_supremacy.ipynb)
# ---------------------------------------------------------------------------

def measure_ZZ(qc, q0, q1, cbit):
    qc.cx(q0, q1)
    qc.measure(q1, cbit)
    qc.cx(q0, q1)

def measure_XI(qc, q0, q1, cbit):
    qc.h(q0)
    qc.measure(q0, cbit)
    qc.h(q0)

def measure_YI(qc, q0, q1, cbit):
    qc.sdg(q0)
    measure_XI(qc, q0, q1, cbit)
    qc.s(q0)

def measure_ZY(qc, q0, q1, cbit):
    qc.sdg(q1)
    qc.h(q1)
    measure_ZZ(qc, q0, q1, cbit)
    qc.h(q1)
    qc.s(q1)

def measure_ZX(qc, q0, q1, cbit):
    qc.h(q1)
    measure_ZZ(qc, q0, q1, cbit)
    qc.h(q1)

In [ ]:
# ---------------------------------------------------------------------------
# Single-qubit MBQC gates
# ---------------------------------------------------------------------------

def add_H(qc, d, a, cbit):
    c = cbit
    measure_XI(qc, a, d, c[0])
    measure_ZY(qc, a, d, c[1])
    measure_YI(qc, a, d, c[2])
    measure_XI(qc, a, d, c[3])
    parity = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[2])
    with qc.if_test(expr.logic_not(parity)):
        qc.y(d)
    qc.x(d)
    qc.reset(a)
    return qc

def add_S(qc, d, a, cbit):
    c = cbit
    measure_XI(qc, a, d, c[0])
    measure_ZZ(qc, a, d, c[1])
    measure_YI(qc, a, d, c[2])
    measure_XI(qc, a, d, c[3])
    parity = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[2])
    with qc.if_test(expr.logic_not(parity)):
        qc.z(d)
    qc.reset(a)
    return qc

def add_SH(qc, d, a, cbit):
    c = cbit
    measure_XI(qc, a, d, c[0])
    measure_ZZ(qc, a, d, c[1])
    measure_ZY(qc, a, d, c[2])
    measure_YI(qc, a, d, c[3])
    measure_XI(qc, a, d, c[4])
    parity_023 = expr.bit_xor(expr.bit_xor(c[0], c[2]), c[3])
    with qc.if_test(parity_023):
        qc.y(d)
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(parity_12):
        qc.z(d)
    qc.reset(a)
    return qc

def add_HS(qc, d, a, cbit):
    c = cbit
    measure_XI(qc, a, d, c[0])
    measure_ZY(qc, a, d, c[1])
    measure_ZZ(qc, a, d, c[2])
    measure_YI(qc, a, d, c[3])
    measure_XI(qc, a, d, c[4])
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(expr.logic_not(parity_12)):
        qc.x(d)
    parity_013 = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[3])
    with qc.if_test(expr.logic_not(parity_013)):
        qc.z(d)
    qc.reset(a)
    return qc

def add_HSH(qc, d, a, cbit):
    c = cbit
    measure_XI(qc, a, d, c[0])
    measure_ZZ(qc, a, d, c[1])
    measure_ZY(qc, a, d, c[2])
    measure_XI(qc, a, d, c[3])
    parity_03 = expr.bit_xor(c[0], c[3])
    with qc.if_test(parity_03):
        qc.y(d)
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(expr.logic_not(parity_12)):
        qc.x(d)
    qc.reset(a)
    return qc

# Supremacy single-qubit gates
def add_sqrtX(qc, d, a, cbit):
    return add_HSH(qc, d, a, cbit)

def add_sqrtY(qc, d, a, cbit):
    qc = add_S(qc, d, a, cbit)
    qc = add_HS(qc, d, a, cbit)
    return qc

In [ ]:
# ---------------------------------------------------------------------------
# MBQC CNOT  —  uses scratch[5], scratch[6], scratch[7]
# (these slots are separate from the single-qubit slots [0:5])
# ---------------------------------------------------------------------------

def add_CNOT(qc, ctrl, anc, targ, cbit):
    qc.reset(anc)
    measure_ZX(qc, ctrl, anc, cbit[5])
    measure_ZX(qc, anc,  targ, cbit[6])
    qc.h(anc)
    qc.measure(anc, cbit[7])
    qc.h(anc)
    qc.reset(anc)
    with qc.if_test(expr.bit_xor(cbit[5], cbit[7])):
        qc.x(targ)
    with qc.if_test((cbit[6], 1)):
        qc.z(ctrl)
    return qc


# ---------------------------------------------------------------------------
# fSim(pi/2, 0) = iSWAP-dagger  (Clifford gate)
#
# Decomposition (from Two_qubit_gate_supremacy.ipynb):
#   fSim(theta, phi) = iSWAPdg(theta) * CPhase(phi)
#   phi = 0  =>  CPhase(0) = I  =>  fSim(pi/2, 0) = iSWAPdg(theta=pi/2)
#
# iSWAPdg(pi/2) circuit:
#   H_c  H_t  CNOT  S_t  CNOT  H_c  H_t  (S^3)_c  (S^3)_t
#   H_c  H_t  CNOT  S_t  CNOT  SH_c  SH_t
# where S^3 = S-dagger and Rz(pi/2) -> S (Clifford)
# ---------------------------------------------------------------------------

def add_fsim_clifford(qc, ctrl, anc, targ, cbit):
    # First half of iSWAPdg
    qc = add_H(qc, ctrl, anc, cbit)
    qc = add_H(qc, targ, anc, cbit)
    qc = add_CNOT(qc, ctrl, anc, targ, cbit)
    qc.s(targ)                        # Rz(pi/2) = S  (Clifford, applied directly)
    qc = add_CNOT(qc, ctrl, anc, targ, cbit)
    qc = add_H(qc, ctrl, anc, cbit)
    qc = add_H(qc, targ, anc, cbit)
    # S-dagger on both qubits, implemented as S*S*S (three S gates)
    for _ in range(3):
        qc = add_S(qc, ctrl, anc, cbit)
    for _ in range(3):
        qc = add_S(qc, targ, anc, cbit)
    # Second half
    qc = add_H(qc, ctrl, anc, cbit)
    qc = add_H(qc, targ, anc, cbit)
    qc = add_CNOT(qc, ctrl, anc, targ, cbit)
    qc.s(targ)                        # Rz(pi/2) = S
    qc = add_CNOT(qc, ctrl, anc, targ, cbit)
    qc = add_SH(qc, ctrl, anc, cbit)
    qc = add_SH(qc, targ, anc, cbit)
    return qc

## 3. Load the Google 53-qubit circuit

In [ ]:
CIRCUIT_FILE = os.path.join(
    GOOGLE_DIR, 'circuit_n53_m12_s0_e0_pABCDCDAB.py'
)

def load_cirq_circuit(path):
    spec = importlib.util.spec_from_file_location('google_circuit', path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.QUBIT_ORDER, mod.CIRCUIT

QUBIT_ORDER, CIRCUIT = load_cirq_circuit(CIRCUIT_FILE)
print(f'Loaded  : {os.path.basename(CIRCUIT_FILE)}')
print(f'Qubits  : {len(QUBIT_ORDER)}')
print(f'Moments : {len(CIRCUIT)}')

## 4. Gate classification

Of the gates that appear in the file:
- **√X** (`X**0.5`) → `add_sqrtX`
- **√Y** (`Y**0.5`) → `add_sqrtY`
- **FSimGate(θ, φ)** → `add_fsim_clifford` (replacing all with fSim(π/2, 0))
- **√W** (`PhasedXPowGate`), **Rz**, and anything else → **dropped**

In [ ]:
def _is_sqrt_X(g):
    return isinstance(g, cirq.XPowGate) and np.isclose(g.exponent, 0.5)

def _is_sqrt_Y(g):
    return isinstance(g, cirq.YPowGate) and np.isclose(g.exponent, 0.5)

def _is_fsim(g):
    return isinstance(g, cirq.FSimGate)

# Audit gate types in the loaded circuit
gate_types = {}
for moment in CIRCUIT:
    for op in moment.operations:
        t = type(op.gate).__name__
        gate_types[t] = gate_types.get(t, 0) + 1

print('Gate type counts in the loaded circuit:')
for k, v in sorted(gate_types.items()):
    print(f'  {k}: {v}')

kept    = sum(v for k, v in gate_types.items()
              if k in ('XPowGate', 'YPowGate', 'FSimGate'))
dropped = sum(v for k, v in gate_types.items()
              if k not in ('XPowGate', 'YPowGate', 'FSimGate'))
print(f'\nKept (sqrt_X + sqrt_Y + FSimGate): {kept}')
print(f'Dropped (sqrt_W, Rz, other):       {dropped}')

## 5. Build the 106-qubit MBQC tetron circuit

- **106 qubits**: T1…T106 (53 data + 53 ancilla tetrons), Qiskit indices 0…105.
- **`c_scratch`**: 8-bit scratch register, reused by every MBQC sub-gate.
- **`c_out`**: 53-bit output register for the final logical-qubit measurements.

All `FSimGate(θ, φ)` instances are translated to `add_fsim_clifford`.
√W gates are dropped silently.

In [ ]:
N_TETRON  = 106
N_DATA    = 53
N_SCRATCH = 8     # scratch bits reused by every MBQC gate call

def build_mbqc_53_circuit(qubit_order, cirq_circuit):
    qr        = QuantumRegister(N_TETRON,  'q')
    c_scratch = ClassicalRegister(N_SCRATCH, 'scratch')
    c_out     = ClassicalRegister(N_DATA,   'out')
    qc        = QuantumCircuit(qr, c_scratch, c_out)

    for moment in cirq_circuit:
        for op in moment.operations:
            g    = op.gate
            grid = [(q.row, q.col) for q in op.qubits]

            if _is_sqrt_X(g):
                d = grid_to_qiskit_index(*grid[0])
                a = grid_to_sq_ancilla_index(*grid[0])
                add_sqrtX(qc, d, a, c_scratch)

            elif _is_sqrt_Y(g):
                d = grid_to_qiskit_index(*grid[0])
                a = grid_to_sq_ancilla_index(*grid[0])
                add_sqrtY(qc, d, a, c_scratch)

            elif _is_fsim(g):
                # Replace any fSim(theta, phi) with fSim(pi/2, 0)
                d1, d2, anc = grid_edge_to_qiskit_indices(grid[0], grid[1])
                add_fsim_clifford(qc, d1, anc, d2, c_scratch)

            else:
                pass  # drop sqrt_W, Rz, etc.

        qc.barrier()

    for i, q in enumerate(qubit_order):
        d = grid_to_qiskit_index(q.row, q.col)
        qc.measure(d, c_out[i])

    return qc


qc_mbqc = build_mbqc_53_circuit(QUBIT_ORDER, CIRCUIT)
print(f'MBQC circuit : {qc_mbqc.num_qubits} qubits, '
      f'{qc_mbqc.num_clbits} clbits, depth = {qc_mbqc.depth()}')

## 6. Build the direct 53-qubit reference circuit

Clifford-native Qiskit instructions throughout so the stabilizer backend
can execute without decomposition overhead:

| Logical gate      | Qiskit instruction             |
|-------------------|--------------------------------|
| √X                | `qc.sx(q)`                     |
| √Y                | `qc.ry(π/2, q)`                |
| fSim(any θ, any φ) | `iSwapGate().inverse()` = iSWAP† |

Rz and √W gates are dropped to match the MBQC circuit.

In [ ]:
iswap_dg = iSwapGate().inverse()    # iSWAP† = fSim(pi/2, 0)

def build_direct_53_circuit(qubit_order, cirq_circuit):
    n         = len(qubit_order)           # 53
    qubit_map = {q: i for i, q in enumerate(qubit_order)}
    qc        = QuantumCircuit(n, n)

    for moment in cirq_circuit:
        for op in moment.operations:
            g     = op.gate
            q_idx = [qubit_map[q] for q in op.qubits]

            if   _is_sqrt_X(g):  qc.sx(q_idx[0])
            elif _is_sqrt_Y(g):  qc.ry(np.pi / 2, q_idx[0])
            elif _is_fsim(g):    qc.append(iswap_dg, q_idx)
            else:                pass  # dropped

        qc.barrier()

    for i in range(n):
        qc.measure(i, i)

    return qc


qc_direct = build_direct_53_circuit(QUBIT_ORDER, CIRCUIT)
print(f'Direct circuit: {qc_direct.num_qubits} qubits, '
      f'{qc_direct.num_clbits} clbits, depth = {qc_direct.depth()}')

## 7. Run both circuits with the stabilizer backend

`AerSimulator(method='stabilizer')` represents each n-qubit Clifford state
in O(n²) memory instead of O(2ⁿ), making 53- and 106-qubit simulation tractable.

### Comparison strategy

If the MBQC feed-forward corrections are correct, the **marginal distribution
over the 53 logical data qubits** from the MBQC circuit should match the
direct-circuit output distribution — regardless of the mid-circuit measurement
trajectory.  We quantify agreement with the **Hellinger fidelity**:

$$H = \left(\sum_{x} \sqrt{p(x)\, q(x)}\right)^2$$

A value close to 1 indicates full agreement; a value near 0 means the
distributions are essentially disjoint.

In [ ]:
N_SHOTS = 2000     # increase for better statistics; stabilizer is very fast
SEED    = 42

backend_stab = AerSimulator(method='stabilizer')

# ── Direct circuit ────────────────────────────────────────────────────────
result_direct  = backend_stab.run(
    qc_direct, shots=N_SHOTS, seed_simulator=SEED
).result()
counts_direct  = result_direct.get_counts()
print(f'Direct : {len(counts_direct)} distinct bitstrings / {N_SHOTS} shots')

# ── MBQC circuit ──────────────────────────────────────────────────────────
result_mbqc_raw = backend_stab.run(
    qc_mbqc, shots=N_SHOTS, seed_simulator=SEED
).result()
counts_mbqc_raw = result_mbqc_raw.get_counts()

# The combined classical string is printed as  "out_bits scratch_bits"
# (space-separated, highest-index register first).
# We keep only the 'out' register bits — the leftmost space-delimited token.
def extract_out_register(counts_combined):
    out_counts = {}
    for key, val in counts_combined.items():
        out_bits = key.split()[0]   # leftmost = c_out
        out_counts[out_bits] = out_counts.get(out_bits, 0) + val
    return out_counts

counts_mbqc = extract_out_register(counts_mbqc_raw)
print(f'MBQC   : {len(counts_mbqc)} distinct logical bitstrings / {N_SHOTS} shots')

In [ ]:
def hellinger_fidelity(counts_a, counts_b, total):
    all_keys = set(counts_a) | set(counts_b)
    inner = sum(
        np.sqrt(counts_a.get(k, 0) / total * counts_b.get(k, 0) / total)
        for k in all_keys
    )
    return float(inner ** 2)


H = hellinger_fidelity(counts_direct, counts_mbqc, N_SHOTS)
print(f'Hellinger fidelity (direct vs MBQC logical output): {H:.6f}')
print(f'Perfect agreement => 1.0 | Unrelated distributions => ~0.0')

In [ ]:
def top_n(counts, n=20):
    return sorted(counts.items(), key=lambda x: -x[1])[:n]

top_direct = top_n(counts_direct, 20)
top_mbqc   = top_n(counts_mbqc,   20)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for ax, data, title in [
    (axes[0], top_direct, f'Direct ({N_SHOTS} shots)'),
    (axes[1], top_mbqc,   f'MBQC logical output ({N_SHOTS} shots)'),
]:
    if data:
        labels, vals = zip(*data)
        short = [l[-10:] for l in labels]   # show last 10 bits for readability
        ax.bar(range(len(vals)), vals)
        ax.set_xticks(range(len(short)))
        ax.set_xticklabels(short, rotation=90, fontsize=7)
    ax.set_title(title)
    ax.set_xlabel('Bitstring (last 10 bits)')
    ax.set_ylabel('Counts')

plt.suptitle(f'Top-20 output bitstrings  |  Hellinger fidelity = {H:.4f}', y=1.02)
plt.tight_layout()
plt.show()

## 8. Notes and caveats

### Why shots-based comparison instead of exact fidelity

The 12-qubit notebook computed exact trajectory-by-trajectory fidelity
$F = \langle\psi_{\rm direct}|\rho_{\rm MBQC,data}|\psi_{\rm direct}\rangle$
from full statevectors.  With 53 qubits the statevector has 2⁵³ entries
(≈ 9 petabytes) — entirely infeasible.  The stabilizer tableau is only
O(n²) but cannot be converted to a statevector.  Shot-based comparison
is the practical alternative: if the feed-forward is correct, the two
output distributions should agree.

### Stronger validation: Pauli shadow tomography (optional)

Append random single-qubit Pauli rotations before measuring, run both
circuits, and estimate single-body and two-body Pauli expectations.
Comparing O(n log n) such values certifies stabilizer-state agreement
up to exponentially small error without ever constructing a statevector.

### Clifford approximation made

The actual `FSimGate` angles in the circuit file are θ ≈ π/2, φ ≈ 0.47–0.57
(approximately π/6 ≈ 0.52).  We replace all of them with fSim(π/2, 0),
which drops the controlled-phase part entirely (φ = 0 makes the CPhase
sub-circuit an identity).  This is a **Clifford projection** of the original
circuit — the comparison is between the Clifford-projected MBQC circuit
and the Clifford-projected direct circuit, not against Google's original
supremacy circuit.

### Gate cost per operation

| Gate           | MBQC parity measurements |
|----------------|-------------------------|
| √X (HSH)       | 4                       |
| √Y (S + HS)    | 4 + 5 = 9               |
| fSim(π/2, 0)   | 2×H + 4×CNOT + 6×S + 2×SH = 2×4 + 4×3 + 6×4 + 2×5 = 58 |

The 8-bit scratch register is the only classical memory required beyond
the 53-bit output register.  Ancilla qubits are reset to |0⟩ within each
sub-gate call.